In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Baselines available in the original notebook
baseline_methods = ['IForest', 'OCSVM', 'AE', 'VAE', 'Deep SVDD', 'MCDSVDD']
class_order = [
    'SLSN', 'SNII', 'SNIa', 'SNIbc',
    'AGN', 'Blazar', 'CV-Nova', 'QSO', 'YSO',
    'CEP', 'DSCT', 'E', 'RRL', 'LPV',
]

# Baseline data: [mean, std] per class in class_order
baseline_data = [
    # IForest
    [[0.640, 0.014], [0.721, 0.021], [0.428, 0.032], [0.490, 0.038],
     [0.573, 0.017], [0.710, 0.009], [0.975, 0.001], [0.468, 0.016], [0.913, 0.003],
     [0.359, 0.007], [0.295, 0.012], [0.469, 0.021], [0.549, 0.033], [0.971, 0.007]],

    # OCSVM
    [[0.577, 0.014], [0.587, 0.014], [0.434, 0.021], [0.492, 0.011],
     [0.532, 0.008], [0.443, 0.002], [0.909, 0.001], [0.517, 0.005], [0.792, 0.005],
     [0.432, 0.004], [0.557, 0.005], [0.555, 0.003], [0.539, 0.004], [0.943, 0.001]],

    # AE
    [[0.736, 0.022], [0.807, 0.021], [0.438, 0.015], [0.537, 0.019],
     [0.701, 0.010], [0.762, 0.006], [0.980, 0.016], [0.443, 0.004], [0.990, 0.001],
     [0.564, 0.024], [0.367, 0.015], [0.864, 0.009], [0.907, 0.015], [0.996, 0.000]],

    # VAE
    [[0.669, 0.015], [0.690, 0.023], [0.404, 0.018], [0.522, 0.025],
     [0.596, 0.007], [0.597, 0.010], [0.849, 0.028], [0.500, 0.009], [0.795, 0.009],
     [0.442, 0.010], [0.417, 0.007], [0.561, 0.007], [0.451, 0.006], [0.936, 0.007]],

    # Deep SVDD
    [[0.644, 0.043], [0.690, 0.043], [0.475, 0.040], [0.507, 0.040],
     [0.496, 0.025], [0.607, 0.044], [0.932, 0.015], [0.411, 0.008], [0.901, 0.022],
     [0.707, 0.027], [0.482, 0.054], [0.636, 0.055], [0.774, 0.068], [0.785, 0.025]],

    # MCDSVDD
    [[0.686, 0.051], [0.828, 0.024], [0.624, 0.039], [0.584, 0.032],
     [0.706, 0.069], [0.512, 0.113], [0.770, 0.127], [0.483, 0.080], [0.854, 0.041],
     [0.858, 0.025], [0.819, 0.015], [0.945, 0.006], [0.953, 0.003], [0.953, 0.008]],
]

baseline_means_df = pd.DataFrame(index=baseline_methods, columns=class_order, dtype=float)
baseline_stds_df = pd.DataFrame(index=baseline_methods, columns=class_order, dtype=float)

for i, method in enumerate(baseline_methods):
    for j, cls in enumerate(class_order):
        baseline_means_df.loc[method, cls] = baseline_data[i][j][0]
        baseline_stds_df.loc[method, cls] = baseline_data[i][j][1]

family_colors = {
    'Transient': '#4C78A8',
    'Stochastic': '#59A14F',
    'Periodic': '#F28E2B',
}

family_by_class = {
    'SLSN': 'Transient', 'SNII': 'Transient', 'SNIa': 'Transient', 'SNIbc': 'Transient',
    'AGN': 'Stochastic', 'Blazar': 'Stochastic', 'CV-Nova': 'Stochastic', 'QSO': 'Stochastic', 'YSO': 'Stochastic',
    'CEP': 'Periodic', 'DSCT': 'Periodic', 'E': 'Periodic', 'RRL': 'Periodic', 'LPV': 'Periodic',
}

# Resolve results root robustly for both notebook and repo-root execution
candidate_roots = [
    # Path('../results_paper/alerce'),
    Path('../results/alerce'),
]
RESULTS_ROOT = next((p.resolve() for p in candidate_roots if p.exists()), None)
if RESULTS_ROOT is None:
    raise FileNotFoundError('Could not locate results/alerce directory.')

print('Results root:', RESULTS_ROOT)
print('Baseline methods loaded:', baseline_methods)

Results root: /home/sguzman/mgscode/thesis/sldnet/results/alerce
Baseline methods loaded: ['IForest', 'OCSVM', 'AE', 'VAE', 'Deep SVDD', 'MCDSVDD']


In [2]:
SIGMA_KEY = 'sigma_0.001'
AUROC_METRIC_KEY = 'roc_auc_log_density_individual'
AUPRC_METRIC_KEY = 'pr_auc_log_density_individual'


def latest_timestamp_dir(fold_dir: Path) -> Path | None:
    run_dirs = sorted([p for p in fold_dir.iterdir() if p.is_dir()])
    return run_dirs[-1] if run_dirs else None


def fold_metric_from_metrics_json(
    metrics_path: Path,
    metric_key: str,
    sigma_key: str | None = None,
    aggregate_field: str = 'mean',
) -> float:
    with metrics_path.open('r') as f:
        metrics = json.load(f)

    if not metrics:
        return np.nan

    latest_epoch_key = max(metrics.keys(), key=lambda epoch: int(epoch))
    latest_epoch_metrics = metrics[latest_epoch_key]
    metric_obj = latest_epoch_metrics.get(metric_key)

    if metric_obj is None:
        return np.nan

    if isinstance(metric_obj, dict):
        if sigma_key is not None and sigma_key in metric_obj:
            return float(metric_obj[sigma_key])
        if aggregate_field in metric_obj:
            return float(metric_obj[aggregate_field])
        return np.nan

    if isinstance(metric_obj, (int, float)):
        return float(metric_obj)

    return np.nan


def collect_ours_fold_metrics(results_root: Path) -> pd.DataFrame:
    rows = []
    for family_dir in sorted([p for p in results_root.iterdir() if p.is_dir()]):
        family = family_dir.name
        for class_dir in sorted([p for p in family_dir.iterdir() if p.is_dir()]):
            cls = class_dir.name
            fold_dirs = sorted(
                [p for p in class_dir.iterdir() if p.is_dir() and p.name.isdigit()],
                key=lambda p: int(p.name),
            )

            for fold_dir in fold_dirs:
                run_dir = latest_timestamp_dir(fold_dir)
                if run_dir is None:
                    continue

                metrics_path = run_dir / 'metrics.json'
                if not metrics_path.exists():
                    continue

                rows.append({
                    'family': family,
                    'class': cls,
                    'fold': int(fold_dir.name),
                    'run_timestamp': run_dir.name,
                    'auroc': fold_metric_from_metrics_json(
                        metrics_path, AUROC_METRIC_KEY, sigma_key=SIGMA_KEY
                    ),
                    'auprc': fold_metric_from_metrics_json(
                        metrics_path, AUPRC_METRIC_KEY, sigma_key=SIGMA_KEY
                    ),
                })

    return pd.DataFrame(rows)


ours_folds_df = collect_ours_fold_metrics(RESULTS_ROOT)
if ours_folds_df.empty:
    raise RuntimeError('No fold-level metrics were found under results/alerce.')

coverage_df = (
    ours_folds_df.groupby(['family', 'class'])['fold']
    .nunique()
    .rename('num_folds')
    .reset_index()
    .sort_values(['family', 'class'])
    .reset_index(drop=True)
)

ours_summary_df = (
    ours_folds_df.groupby(['family', 'class'])
    .agg(
        auroc_mean=('auroc', 'mean'),
        auroc_std=('auroc', 'std'),
        auprc_mean=('auprc', 'mean'),
        auprc_std=('auprc', 'std'),
        num_folds=('fold', 'nunique'),
    )
    .reset_index()
)

for col in ['auroc_std', 'auprc_std']:
    ours_summary_df[col] = ours_summary_df[col].fillna(0.0)

available_classes = ours_summary_df['class'].tolist()
ordered_available_classes = [c for c in class_order if c in available_classes]
extra_classes = sorted(set(available_classes) - set(class_order))
ordered_classes = ordered_available_classes + extra_classes

ours_summary_df['class'] = pd.Categorical(ours_summary_df['class'], categories=ordered_classes, ordered=True)
ours_summary_df = ours_summary_df.sort_values('class').reset_index(drop=True)

missing_from_results = [c for c in class_order if c not in ordered_classes]

print(f'Using sigma key: {SIGMA_KEY} from the last available epoch in metrics.json')
print('Fold coverage per family/class:')
display(coverage_df)

if missing_from_results:
    print('Classes in baseline but not found in results:', missing_from_results)

print('\nOurs summary (mean/std over folds):')
display(ours_summary_df[['family', 'class', 'num_folds', 'auroc_mean', 'auroc_std', 'auprc_mean', 'auprc_std']])

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
plot_df = ours_summary_df.copy()
plot_df['color'] = plot_df['family'].map(family_colors).fillna('#999999')

x = np.arange(len(plot_df))
class_labels = plot_df['class'].tolist()

mcd_means = [baseline_means_df.loc['MCDSVDD', c] if c in baseline_means_df.columns else np.nan for c in class_labels]
mcd_stds = [baseline_stds_df.loc['MCDSVDD', c] if c in baseline_stds_df.columns else np.nan for c in class_labels]

family_handles = [Patch(facecolor=color, edgecolor='none', label=family) for family, color in family_colors.items()]

# AUROC (sigma_0.001): Ours + MCDSVDD reference
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(
    x - 0.18,
    plot_df['auroc_mean'].values,
    yerr=plot_df['auroc_std'].values,
    width=0.36,
    color=plot_df['color'].values,
    alpha=0.72,
    capsize=6,
    ecolor='black',
    linewidth=0,
    label='Ours',
)
ax.bar(
    x + 0.18,
    mcd_means,
    yerr=mcd_stds,
    width=0.36,
    color='lightgray',
    alpha=0.95,
    capsize=6,
    ecolor='gray',
    linewidth=0,
    label='MCDSVDD',
)

ax.set_title(f'AUROC by Class (sigma={SIGMA_KEY}, Ours vs MCDSVDD baseline)')
ax.set_ylabel('AUROC')
ax.set_xlabel('Class')
ax.set_xticks(x)
ax.set_xticklabels(class_labels, rotation=45, ha='right')
ax.set_ylim(0.0, 1.0)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

legend_handles = family_handles + [Patch(facecolor='lightgray', edgecolor='none', label='MCDSVDD')]
ax.legend(handles=legend_handles, title='Family / Reference', loc='upper left')
plt.tight_layout()
plt.show()

# AUPRC (sigma_0.001): Ours only
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(
    x,
    plot_df['auprc_mean'].values,
    yerr=plot_df['auprc_std'].values,
    width=0.62,
    color=plot_df['color'].values,
    alpha=0.72,
    capsize=6,
    ecolor='black',
    linewidth=0,
    label='Ours',
)

ax.set_title(f'AUPRC by Class (sigma={SIGMA_KEY}, Ours)')
ax.set_ylabel('AUPRC')
ax.set_xlabel('Class')
ax.set_xticks(x)
ax.set_xticklabels(class_labels, rotation=45, ha='right')
ax.set_ylim(0.0, 1.0)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

ax.legend(handles=family_handles, title='Family', loc='upper left')
plt.tight_layout()
plt.show()

# Compact numeric view
display(plot_df[['family', 'class', 'auroc_mean', 'auroc_std', 'auprc_mean', 'auprc_std']])